In [2]:
import torch
import numpy as np
import gc
import pandas as pd

def get_sparse_row(A_sparse, row_idx, num_cols):
    """
    Извлекает row_idx-ю строку разреженного тензора A_sparse (формат COO)
    и возвращает её в виде плотного вектора размера (num_cols,).
    """
    indices = A_sparse._indices()  # [2, nnz]
    values = A_sparse._values()    # [nnz]
    mask = (indices[0] == row_idx)
    col_indices = indices[1, mask]
    row_values = values[mask]
    dense_row = torch.zeros(num_cols, device=A_sparse.device, dtype=A_sparse.dtype)
    dense_row.index_put_((col_indices,), row_values)
    return dense_row

def extract_sparse_columns(A_sparse, col_indices, num_rows, r):
    """
    Извлекает подматрицу, состоящую из выбранных столбцов,
    и переиндексирует их в диапазон 0...r-1.
    
    Аргументы:
      A_sparse (torch.sparse_coo_tensor): исходная матрица (n x m)
      col_indices (torch.Tensor): 1D тензор из r индексов столбцов
      num_rows (int): число строк n
      r (int): число выбранных столбцов
      
    Возвращает:
      A_cols (torch.sparse_coo_tensor): разреженная матрица (n x r)
    """
    orig_indices = A_sparse._indices()  # [2, nnz]
    orig_values = A_sparse._values()
    
    # Фильтруем ненулевые элементы, принадлежащие выбранным столбцам
    mask = torch.zeros(orig_indices.shape[1], dtype=torch.bool, device=A_sparse.device)
    for ci in col_indices.cpu():
        mask |= (orig_indices[1] == ci.item())
    
    new_indices = orig_indices[:, mask].clone()
    new_values = orig_values[mask].clone()
    
    # Создаем маппинг: оригинальный номер столбца -> новый номер (0...r-1)
    mapping = -torch.ones(A_sparse.shape[1], dtype=torch.long, device=A_sparse.device)
    mapping[col_indices] = torch.arange(r, device=A_sparse.device)
    
    new_indices[1] = mapping[new_indices[1]]
    
    A_cols = torch.sparse_coo_tensor(new_indices, new_values, size=(num_rows, r)).coalesce()
    return A_cols

def maxvol_gpu(A, e=1.05, k=100):
    """
    Точный алгоритм maxvol для плотной "tall" матрицы A (n x r, n > r) на GPU.
    
    Аргументы:
      A (torch.Tensor): плотная матрица размера (n x r) на GPU.
      e (float): параметр точности (>=1). При e=1 алгоритм идёт до истинной сходимости.
      k (int): максимальное число итераций.
      
    Возвращает:
      I (torch.Tensor): 1D тензор длины r с индексами выбранных строк.
      B (torch.Tensor): коэффициентная матрица (n x r), такая что A = B @ A[I, :].
    """
    n, r = A.shape
    if n <= r:
        raise ValueError("Матрица A должна быть 'tall': n > r")
    
    # LU-разложение: A = P L U
    LU, pivots = torch.lu(A)
    P, L, U = torch.lu_unpack(LU, pivots)
    # Инициализация: для каждого столбца из первых r столбцов P выбираем индекс строки, где стоит 1
    I = torch.argmax(P[:, :r], dim=0).clone()  # I имеет размер (r,)
    
    # Решаем U^T x = A^T:
    # U^T является нижнетреугольной, поэтому решаем:
    Q = torch.linalg.solve_triangular(U.T, A.T, upper=False, left=True, unitriangular=False)
    
    # Решаем (L[:r, :])^T y = Q:
    # L[:r, :].T является верхнетреугольной (так как L – нижнетреугольная)
    Y = torch.linalg.solve_triangular(L[:r, :].T, Q, upper=True, left=True, unitriangular=True)
    B = Y.T  # Тогда A = B @ A[I, :]
    
    for _ in range(k):
        absB = torch.abs(B)
        max_idx = torch.argmax(absB)
        i = max_idx // r   # кандидат на замену строки
        j = max_idx % r    # номер столбца в B
        if absB[i, j] <= e:
            break
        I[j] = i  # обновляем индекс для j-го столбца
        bj = B[:, j].clone()
        bi = B[i, :].clone()
        bi[j] = bi[j] - 1.0
        B = B - torch.outer(bj, bi) / B[i, j]
    
    return I, B


def maxvol_sparse_gpu(A_sparse, r, e=1.05, k=100):
    """
    Точный алгоритм maxvol для больших разреженных матриц на GPU.
    Выполняется выбор r столбцов по L2-норме, извлекается подматрица A_cols (n x r) в dense виде,
    затем применяется maxvol для выбора r строк, что дает подматрицу пересечения A[I, J] с максимальным объёмом.
    
    Аргументы:
      A_sparse (torch.sparse_coo_tensor): исходная разреженная матрица размера (n x m) на GPU.
      r (int): требуемый размер подматрицы (r x r).
      e (float): параметр точности для maxvol.
      k (int): максимальное число итераций maxvol.
      
    Возвращает:
      submatrix (torch.Tensor): пересечение выбранных строк и столбцов (dense, r x r).
      J (torch.Tensor): 1D тензор индексов выбранных столбцов (размер r).
      I (torch.Tensor): 1D тензор индексов выбранных строк (размер r).
    """
    n, m = A_sparse.shape
    # Вычисляем L2-нормы столбцов с учетом разреженности
    indices = A_sparse._indices()  # [2, nnz]
    values = A_sparse._values()
    col_norms = torch.zeros(m, device=A_sparse.device, dtype=A_sparse.dtype)
    col_norms = col_norms.scatter_add(0, indices[1], values**2)
    col_norms = torch.sqrt(col_norms)
    
    # Выбираем r столбцов с наибольшей нормой
    _, J = torch.topk(col_norms, r)
    J, _ = torch.sort(J)  # сортировка для согласованности
    
    # Извлекаем подматрицу из выбранных столбцов и переводим в плотный формат (n x r)
    A_cols_sparse = extract_sparse_columns(A_sparse, J, n, r)
    A_cols = A_cols_sparse.to_dense()
    
    # Применяем точный maxvol для плотной "tall" матрицы A_cols
    I, _ = maxvol_gpu(A_cols, e=e, k=k)
    
    # Пересечение – это A_cols[I, :] (dense, r x r)
    submatrix = A_cols[I, :]
    
    return submatrix, J, I

In [3]:
# Генерация разреженной матрицы в NumPy с рейтингами от 1 до 5:
n, m, r = 12000, 7000, 5000
density = 0.1
mask_np = np.random.binomial(1, density, size=(n, m)).astype(np.int8)
rows_np, cols_np = np.nonzero(mask_np)
ratings = np.random.randint(1, 6, size=(n, m), dtype=np.int8)
values_np = ratings * mask_np

del mask_np
del ratings
gc.collect()

20

In [2]:
values = pd.read_csv('UI_data_2.csv')
values.drop(columns=['userId'], inplace=True)
values = (values.values).astype(np.int8)
rows_np, cols_np = np.nonzero(values)

In [4]:
n, m = values.shape
print(n,m)
r = m//2

25607 7706


In [61]:
indices[0].shape

torch.Size([3012677])

In [5]:
# Переводим в PyTorch sparse тензор на GPU:
indices = torch.tensor([rows_np, cols_np], dtype=torch.long, device="cuda")
values_nonzero = values[rows_np, cols_np]
values_torch = torch.tensor(values_nonzero, dtype=torch.float32, device="cuda")
A_sparse = torch.sparse_coo_tensor(indices, values_torch, size=(n, m)).coalesce()

# Применяем maxvol для больших разреженных матриц:
submatrix, cols, rows = maxvol_sparse_gpu(A_sparse, r, e=1.05, k=100)

print("Размер подматрицы:", submatrix.shape)
print("Индексы выбранных столбцов:", cols.cpu().numpy())
print("Индексы выбранных строк:", rows.cpu().numpy())

C:\Users\tir1\AppData\Local\Temp\ipykernel_7732\2290230158.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  indices = torch.tensor([rows_np, cols_np], dtype=torch.long, device="cuda")
c:\Programs\Python\3.13\Lib\site-packages\torch\functional.py:2140: UserWarning: torch.lu is deprecated in favor of torch.linalg.lu_factor / torch.linalg.lu_factor_ex and will be removed in a future PyTorch release.
LU, pivots = torch.lu(A, compute_pivots)
should be replaced with
LU, pivots = torch.linalg.lu_factor(A, compute_pivots)
and
LU, pivots, info = torch.lu(A, compute_pivots, get_infos=True)
should be replaced with
LU, pivots, info = torch.linalg.lu_factor_ex(A, compute_pivots) (Triggered internally at C:\actions-runner\_work\pytorch\pytorc

Размер подматрицы: torch.Size([3853, 3853])
Индексы выбранных столбцов: [   2    4    5 ... 7700 7702 7705]
Индексы выбранных строк: [  469    53  1074 ...  1388 11670 24272]


In [7]:
# Переводим индексы в numpy
cols_np = cols.cpu().numpy()   # J — индексы столбцов
rows_np = rows.cpu().numpy()   # I — индексы строк

# Вырезаем из оригинальной плотной матрицы values_np:
C = values[:, cols_np]      # C: (n x r)
R = values[rows_np, :]      # R: (r x m)
U = torch.inverse(submatrix).cpu().numpy()  # U: (r x r)

# Приближённая матрица:
approx = C @ U @ R  # размер: (n x m)
approx_rounded = np.clip(np.rint(approx), 0, 5).astype(np.int32)


# Ошибка восстановления (до округления):
rel_error = np.linalg.norm(values - approx) / np.linalg.norm(values)
print(f"Относительная ошибка (без округления): {rel_error:.6f}")

# Ошибка после округления:
rel_error_rounded = np.linalg.norm(values - approx_rounded) / np.linalg.norm(values)
print(f"Относительная ошибка (с округлением): {rel_error_rounded:.6f}")

Относительная ошибка (без округления): 3.355189
Относительная ошибка (с округлением): 1.620453


In [9]:
test_mat = U@R

In [15]:
test_c = np.random.randint(0, 5, [1,3853])

In [16]:
test_c = test_c * np.random.binomial(1, 0.1, [1,3853])

In [24]:
a = test_c @ test_mat
np.clip(np.rint(a),0,5)

array([[ 0.,  0., -0., ...,  2.,  0., -0.]])

In [18]:
C.shape

(25607, 3853)

In [35]:
import numpy as np
import torch
import gc

def block_maxvol_gpu(A, e=1.05, k=100, chunk_size=10000):
    """
    Блочная реализация maxvol для dense "tall" матрицы A (n x r) на GPU с обработкой по батчам.
    
    A должна быть на GPU и иметь форму (n x r), где n >> r.
    
    Алгоритм:
      1. Инициализация: I = [0, 1, ..., r-1], sub = A[I, :] и inv_sub = inv(sub).
      2. Для до k итераций:
           a. Проходим по A блоками по строкам (batch_size = chunk_size).
           b. Для каждого батча вычисляем B_block = (блок A) @ inv_sub.
           c. Находим максимум по модулю в B_block. Если максимум <= e, завершаем.
           d. Иначе, для найденного максимума (в позиции (i, j) в B, где i – глобальный индекс строки, j – номер столбца):
              обновляем I[j] = i и пересчитываем inv_sub = inv(A[I, :]).
      3. Возвращаем I и inv_sub.
    """
    n, r = A.shape
    # Инициализация: первые r строк
    I = torch.arange(r, device=A.device)
    sub = A[I, :]  # (r x r)
    inv_sub = torch.pinverse(sub)
    
    for iteration in range(k):
        max_val = 0.0
        max_row = None
        max_col = None
        # Обходим A батчами по строкам
        for start in range(0, n, chunk_size):
            end = min(start + chunk_size, n)
            block = A[start:end, :]  # размер (block_size x r)
            # Вычисляем коэффициенты: B_block = block @ inv_sub
            B_block = torch.matmul(block, inv_sub)  # (block_size x r)
            absB_block = torch.abs(B_block)
            local_max_val, local_max_idx = torch.max(absB_block.view(-1), 0)
            if local_max_val.item() > max_val:
                max_val = local_max_val.item()
                local_row = local_max_idx // r
                local_col = local_max_idx % r
                max_row = start + local_row.item()  # глобальный индекс строки
                max_col = local_col.item()
        # Если максимум меньше или равен порогу, алгоритм сходится
        if max_val <= e:
            #print(f"Converged after {iteration} iterations with max coefficient {max_val:.4f}")
            break
        # Обновляем выбранный индекс для столбца max_col
        I[max_col] = max_row
        # Пересчитываем инверсию подматрицы
        sub = A[I, :]  # (r x r)
        inv_sub = torch.inverse(sub)
    return I, inv_sub

# ----------------- Остальные вспомогательные функции -----------------

def compute_col_norms_sparse(A_sparse, chunk_size=10000):
    """
    Вычисляет L2-нормы столбцов разреженной матрицы A_sparse по частям (батчами).
    """
    n, m = A_sparse.shape
    col_norms_sq = torch.zeros(m, device=A_sparse.device, dtype=A_sparse.dtype)
    A_coalesced = A_sparse.coalesce()
    indices = A_coalesced._indices()  # shape: [2, nnz]
    values = A_coalesced._values()
    
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        mask = (indices[0] >= start) & (indices[0] < end)
        if mask.sum() == 0:
            continue
        batch_cols = indices[1, mask]
        batch_values = values[mask]
        col_norms_sq = col_norms_sq.scatter_add(0, batch_cols, batch_values**2)
    
    return torch.sqrt(col_norms_sq)

def extract_sparse_columns(A_sparse, col_indices, num_rows, r):
    """
    Извлекает подматрицу, состоящую из выбранных столбцов col_indices из A_sparse и переиндексирует их в диапазон 0...r-1.
    """
    orig_indices = A_sparse._indices()
    orig_values = A_sparse._values()
    
    mask = torch.zeros(orig_indices.shape[1], dtype=torch.bool, device=A_sparse.device)
    for ci in col_indices.cpu():
        mask |= (orig_indices[1] == ci.item())
        
    new_indices = orig_indices[:, mask].clone()
    new_values = orig_values[mask].clone()
    
    mapping = -torch.ones(A_sparse.shape[1], dtype=torch.long, device=A_sparse.device)
    mapping[col_indices] = torch.arange(r, device=A_sparse.device)
    new_indices[1] = mapping[new_indices[1]]
    
    A_cols = torch.sparse_coo_tensor(new_indices, new_values, size=(num_rows, r)).coalesce()
    return A_cols

def sparse_to_dense_chunked(A_sparse, chunk_size=10000):
    """
    Преобразует разреженную матрицу A_sparse (COO) в плотную, обрабатывая строки батчами.
    """
    n, r = A_sparse.shape
    dense_chunks = []
    A_coalesced = A_sparse.coalesce()
    indices = A_coalesced._indices()
    values = A_coalesced._values()
    
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        batch_mask = (indices[0] >= start) & (indices[0] < end)
        if batch_mask.sum() == 0:
            dense_chunk = torch.zeros((end - start, r), device=A_sparse.device, dtype=A_sparse.dtype)
        else:
            batch_indices = indices[:, batch_mask].clone()
            batch_values = values[batch_mask].clone()
            batch_indices[0] = batch_indices[0] - start
            dense_chunk = torch.zeros((end - start, r), device=A_sparse.device, dtype=A_sparse.dtype)
            dense_chunk.index_put_((batch_indices[0], batch_indices[1]), batch_values, accumulate=True)
        dense_chunks.append(dense_chunk)
    return torch.cat(dense_chunks, dim=0)

def free_memory(*vars):
    for v in vars:
        del v
    gc.collect()

Синтетика

In [10]:
# ----------------- CUR-разложение и оценка аппроксимации -----------------

# Параметры исходной матрицы
n = 70000   # число строк
m = 10000    # число столбцов
r = 5000     # требуемый размер подматрицы (r x r) для CUR
density = 0.1

# Генерация разреженной матрицы в NumPy:
mask_np = np.random.binomial(1, density, size=(n, m)).astype(np.uint8)
ratings = np.random.randint(1, 6, size=(n, m)).astype(np.int8)
values_np = (ratings * mask_np).astype(np.float32)  # оригинальная матрица, float32

free_memory(mask_np, ratings)

Реальные значения

In [42]:
import pandas as pd
values = pd.read_csv('UI_data_2.csv')
values.drop(columns=['userId'], inplace=True)
values_np = (values.values).astype(np.int8)
free_memory(values)

In [45]:
n, m = values_np.shape
r=1000

In [46]:
# Создаём разрежённое представление в PyTorch (COO) на GPU:
rows_np, cols_np = np.nonzero(values_np)
values_nonzero = values_np[rows_np, cols_np]
indices = torch.tensor([rows_np, cols_np], dtype=torch.long, device="cuda")
values_torch = torch.tensor(values_nonzero, dtype=torch.float32, device="cuda")  # используем float32 для точности
A_sparse = torch.sparse_coo_tensor(indices, values_torch, size=(n, m)).coalesce()

free_memory(rows_np, cols_np, values_nonzero, indices, values_torch)

# Вычисляем нормы столбцов батчами и выбираем r столбцов с наибольшей нормой:
col_norms = compute_col_norms_sparse(A_sparse, chunk_size=1000)
_, J = torch.topk(col_norms, r)
J, _ = torch.sort(J)

# Извлекаем подматрицу выбранных столбцов (A_cols_sparse) и переводим её в dense по батчам:
A_cols_sparse = extract_sparse_columns(A_sparse, J, n, r)
A_cols = sparse_to_dense_chunked(A_cols_sparse, chunk_size=1000)

# Применяем блочный maxvol для выбора r строк:
I, inv_sub = block_maxvol_gpu(A_cols, e=1.05, k=10, chunk_size=1000)

In [47]:
# Итоговая подматрица пересечения (dense, r x r):
submatrix = A_cols[I, :]

# Переводим выбранные индексы в NumPy:
cols_np_selected = J.cpu().numpy()   # выбранные столбцы
rows_np_selected = I.cpu().numpy()     # выбранные строки

# Формируем матрицы CUR на CPU:
C = values_np[:, cols_np_selected]      # C: (n x r)
R = values_np[rows_np_selected, :]       # R: (r x m)
U = np.linalg.inv(submatrix.cpu().numpy())  # U: (r x r)

# Восстанавливаем приближение:
approx = C @ U @ R

# Считаем относительную ошибку восстановления по норме Фробениуса:
rel_error = np.linalg.norm(values_np - approx, ord='fro') / np.linalg.norm(values_np, ord='fro')
print(f"Относительная ошибка восстановления: {rel_error:.6f}")

# Если требуется, можно округлить approx до целых (и обрезать до [1,5]):
approx_rounded = np.clip(np.rint(approx), 0, 5)
rel_error_rounded = np.linalg.norm(values_np - approx_rounded, ord='fro') / np.linalg.norm(values_np, ord='fro')
print(f"Относительная ошибка восстановления (с округлением): {rel_error_rounded:.6f}")

free_memory(inv_sub, submatrix, approx, approx_rounded)

Относительная ошибка восстановления: 3.949966
Относительная ошибка восстановления (с округлением): 1.746817


In [14]:
dataframe = pd.read_csv('UI_data_2.csv')

In [17]:
dataframe.drop(columns=['userId'], inplace=True)

In [18]:
dataframe.head()

,'Til There Was You (1997),"'burbs, The (1989)",(500) Days of Summer (2009),*batteries not included (1987),...And Justice for All (1979),10 (1979),10 Cloverfield Lane (2016),10 Things I Hate About You (1999),"10,000 BC (2008)",100 Girls (2000),...,Zorba the Greek (Alexis Zorbas) (1964),Zulu (1964),[REC] (2007),[REC]² (2009),eXistenZ (1999),"tick, tick...BOOM! (2021)",xXx (2002),xXx: Return of Xander Cage (2017),xXx: State of the Union (2005),¡Three Amigos! (1986)
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,3,0,...,0,0,0,0,0,0,2,0,0,0


In [22]:
harry_potter_movies = [
    "Harry Potter and the Half-Blood Prince (2009)",
    "Harry Potter and the Deathly Hallows: Part 2 (2011)",
    "Harry Potter and the Deathly Hallows: Part 1 (2010)",
    "Harry Potter and the Prisoner of Azkaban (2004)"
]

target_movies = [
    "Friends with Benefits (2011)",
    "Ten Commandments, The (1956)",
    "Thin Man, The (1934)",
    "Beauty and the Beast (2017)",
    "Bedazzled (2000)"
]

# Шаг 1: Найти пользователей с оценкой 5 по всем фильмам о Гарри Поттере
hp_users = dataframe[(dataframe[harry_potter_movies] == 5).all(axis=1)]

# Шаг 2: Взять оценки целевых фильмов, заменив 0 на NaN (для игнорирования в среднем)
target_ratings = hp_users[target_movies].replace(0, pd.NA)

# Шаг 3: Посчитать среднее по каждому целевому фильму
mean_ratings = target_ratings.mean()

# Шаг 4: Вывести результат в нужном формате
print("Средние оценки для целевых фильмов:")
for movie, rating in mean_ratings.items():
    print(f"{movie} - {rating:.2f}" if not pd.isna(rating) else f"{movie} - Нет данных")

Средние оценки для целевых фильмов:
Friends with Benefits (2011) - 3.88
Ten Commandments, The (1956) - 3.88
Thin Man, The (1934) - 4.20
Beauty and the Beast (2017) - 4.10
Bedazzled (2000) - 3.65


In [5]:
films = list(dataframe.columns)

In [6]:
films[1646]

'Elite Squad (Tropa de Elite) (2007)'

In [7]:
from difflib import get_close_matches

In [ ]:
from fuzzywuzzy import process
match = process.extractOne("Harry Potter and the Half-Blood Prince", films)

In [12]:
match

('Harry Potter and the Half-Blood Prince (2009)', 100)

In [10]:
films.index(match[0])

2370

In [23]:
films[2370]

'Harry Potter and the Half-Blood Prince (2009)'

In [29]:
dataframe[films[2370]].astype(bool).sum()

np.int64(18652)

In [40]:
get_close_matches('Harry Potter and the Half-Blood Prince', films, n=100, cutoff=0.3)

['Harry Potter and the Half-Blood Prince (2009)',
 'Harry Potter and the Deathly Hallows: Part 2 (2011)',
 'Harry Potter and the Deathly Hallows: Part 1 (2010)',
 'Harry Potter and the Prisoner of Azkaban (2004)',
 'Harry Potter and the Order of the Phoenix (2007)',
 'Harry Potter and the Goblet of Fire (2005)',
 'Harry Potter and the Chamber of Secrets (2002)',
 'Monty Python and the Holy Grail (1975)',
 'Harry and the Hendersons (1987)',
 'Monty Python Live at the Hollywood Bowl (1982)',
 'Quatermass and the Pit (1967)',
 'Lars and the Real Girl (2007)',
 'Davy Crockett, King of the Wild Frontier (1955)',
 'Indiana Jones and the Temple of Doom (1984)',
 'Harley Davidson and the Marlboro Man (1991)',
 'Play it to the Bone (1999)',
 'Father of the Bride (1991)',
 'Father of the Bride (1950)',
 'Raya and the Last Dragon (2021)',
 'Kubo and the Two Strings (2016)',
 'Pat Garrett and Billy the Kid (1973)',
 'Valerian and the City of a Thousand Planets (2017)',
 'Mrs. Parker and the Viciou